# Layer 6: Infrastructure Design Translation


In [ ]:
import os
import json
import numpy as np
import pandas as pd

from mda.loader import load_trades, load_orderbook, load_orderbook_updates
from mda.timestamps import add_ts_columns
from mda import EXCHANGES, DATA_DIR

# Layer summaries
from mda.layer1.rates import layer1_summary
from mda.layer2.sizes import compute_size_distribution, compute_notional_distribution
from mda.layer2.volleys import detect_volleys, volley_size_distribution
from mda.layer2.aggressor import compute_aggressor_imbalance, compute_overall_buy_ratio
from mda.layer3.quantisation import resolution_report
from mda.layer3.execution_rate import compute_execution_rate, execution_rate_stats
from mda.layer3.latency import compute_feed_latency, latency_stats
from mda.layer4.bbo import compute_bbo, spread_stats
from mda.layer4.update_rates import compute_update_rate_by_depth

# Design spec
from mda.layer6.design_specs import compute_specs, write_design_spec


In [ ]:
DATA_DIR = "/data/parquet"
REPORTS_DIR = "/data/reports"
os.makedirs(REPORTS_DIR, exist_ok=True)


## Load pre-computed layer results


In [ ]:
# Load trades once and run all layer summaries
df = load_trades(DATA_DIR, exchanges=EXCHANGES, add_ts_cols=True)
print("Trades loaded:", df.shape)

# Layer 1
l1_results = layer1_summary(df)
l1 = {
    "rate_pcts": l1_results["rate_pcts"],
    "peak_rate": float(l1_results["rate_pcts"].max().max()),
    "p99_rate": float(l1_results["rate_pcts"].loc["p99"].max())
    if "p99" in l1_results["rate_pcts"].index
    else None,
    "total_messages": len(df),
    "n_exchanges": df["exchange"].nunique(),
}

# Layer 2
size_dist = compute_size_distribution(df)
notional = compute_notional_distribution(df)
_, volley_stats = detect_volleys(df)
vsizes = volley_size_distribution(volley_stats)
buy_ratio = compute_overall_buy_ratio(df)
l2 = {
    "size_dist": size_dist,
    "notional": notional,
    "volley_sizes": vsizes,
    "buy_ratio": buy_ratio,
}

# Layer 3
res = resolution_report(df)
exec_rate = compute_execution_rate(df)
exec_stats = execution_rate_stats(exec_rate)
df_lat = compute_feed_latency(df)
lat_stats_result = latency_stats(df_lat)
l3 = {
    "resolution": res,
    "exec_stats": exec_stats,
    "lat_stats": lat_stats_result,
}

# Layer 4 (BBO only for speed)
ob = load_orderbook(DATA_DIR, exchanges=EXCHANGES, bbo_only=True)
bbo = compute_bbo(ob)
s = spread_stats(bbo)
ob_full = load_orderbook(DATA_DIR, exchanges=EXCHANGES, bbo_only=False, max_level=20)
update_rate = compute_update_rate_by_depth(ob_full)
l4 = {
    "spread_stats": s,
    "update_rate": update_rate,
}

print("All layer summaries computed.")


In [ ]:
specs = compute_specs(l1, l2, l3, l4)

print("Infrastructure design specifications:")
print(json.dumps(specs, indent=2, default=str))


In [ ]:
spec_path = os.path.join(REPORTS_DIR, "design_spec.md")
write_design_spec(specs, spec_path)
print(f"Design spec written to: {spec_path}")


In [ ]:
# Display the written markdown spec
with open(spec_path, "r") as f:
    spec_md = f.read()

from IPython.display import Markdown, display
display(Markdown(spec_md))


## Summary Table


In [ ]:
summary_df = pd.DataFrame(
    list(specs.items()),
    columns=["Parameter", "Value"]
)
summary_df
